# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
from langchain_community.document_loaders import PyPDFLoader

def load_pdf_text(file_path: str) -> str:
    loader = PyPDFLoader(file_path)
    docs = loader.load()

    print(f"Number of Pages: {len(docs)}")

    document_text = ""
    for page in docs:
        document_text += page.page_content + "\n"

    return document_text

file_path="../05_src/documents/ai_report_2025.pdf"
document_text = load_pdf_text(file_path)

Number of Pages: 26


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [3]:
from pydantic import BaseModel

class DocumentSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

In [4]:
developer_instructions_prompt = """
You are an AI system that produces structured output using a Pydantic BaseModel.
Follow these rules strictly:

1. Use ONLY the provided context to extract:
   - Author
   - Title
   - Relevance (one paragraph explaining why this article is relevant
     for an AI professional in their professional development)
   - Summary (no more than 1000 tokens)
   - Tone (the tone used to write the summary)

2. The summary MUST:
   - Use ONLY information explicitly stated in the context.
   - Preserve the order of ideas as they appear in the document.
   - Avoid interpretation, inference, or added conclusions.
   - Avoid adding themes, emphasis, or evaluative language.
   - Avoid reorganizing or reframing the content.
   - Avoid transitions such as "in conclusion", "a major theme", etc.
   - Be a STRICT extractive summary.

4. Output MUST be a valid JSON object that matches the Pydantic model fields exactly.

5. Do NOT fabricate or guess information that is not present in the context.
   If a field is missing in the document, set it as "Unknown".

6. Do NOT invent token counts. use the values returned by the model
   for both input and output.

7. You must use a model that is NOT in the GPT‑5 family.
"""

user_prompt_template = """
Context:
{context}

Please analyze the context and produce the structured output.
Use the tone: {tone}.
"""

In [5]:
from langchain.chat_models import init_chat_model
import os

model = init_chat_model(
    "gpt-4o-mini",
    model_provider="openai",
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
    api_key="any value"
)

model_with_structure_output = model.with_structured_output(
    DocumentSummary,
    include_raw=True
)

tone="Neutral Extractive Tone"

formatted_instructions = developer_instructions_prompt.format(
    tone=tone
)

user_prompt = user_prompt_template.format(
    context=document_text[:25000],
    tone=tone
)

full_prompt = f"""
{formatted_instructions}

{user_prompt}
"""

raw_response = model_with_structure_output.invoke(full_prompt)

result_object: DocumentSummary = raw_response["parsed"]

usage = raw_response["raw"].response_metadata.get("token_usage", {})

result_object.InputTokens = usage.get("prompt_tokens", 0)
result_object.OutputTokens = usage.get("completion_tokens", 0)


print(result_object.model_dump_json(indent=4))


{
    "Author": "Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari",
    "Title": "The GenAI Divide: State of AI in Business 2025",
    "Relevance": "This article explores the stark contrasts in the implementation and impact of Generative AI in enterprises, providing critical insights on the success and failure rates of AI initiatives. Understanding these dynamics is essential for AI professionals aiming to refine their strategies, enhance adoption rates, and drive successfully measurable outcomes from AI investments.",
    "Summary": "The report reveals that despite substantial investments in Generative AI (GenAI), an overwhelming majority of organizations (95%) report no return on investment. The findings highlight a significant disparity, termed the GenAI Divide, between those who are effectively utilizing AI tools and those who are not. While tools such as ChatGPT and Copilot have seen widespread exploration and adoption, many organizations still face challenges in ac

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [6]:
from deepeval.test_case import LLMTestCase
from deepeval.test_case import LLMTestCaseParams
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.models import DeepEvalBaseLLM

In [7]:
class GatewayLLM(DeepEvalBaseLLM):
    def __init__(self, model, model_name: str):
        self.model = model
        self.model_name = model_name

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        response = self.model.invoke(prompt)
        return response.content

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self) -> str:
        return self.model_name


In [8]:
gateway_llm = GatewayLLM(model, model_name="gpt-4o-mini")

In [9]:
summarization_questions = [
    "Does the summary capture all major themes from the document?",
    "Does the summary avoid adding information not present in the document?",
    "Is the summary concise while still being complete?",
    "Does the summary accurately reflect the author's intent?",
    "Is the summary logically structured and easy to follow?"
]

coherence_questions = [
    "Is the summary logically organized?",
    "Are transitions between ideas smooth and clear?",
    "Does the summary avoid contradictions?",
    "Is the writing easy to understand?",
    "Does the summary maintain a consistent point of view?"
]

tonality_questions = [
    "Does the summary maintain a Neutral Extractive Tone?",
    "Is the tone consistent throughout the summary?",
    "Does the summary avoid emotional or biased language?",
    "Is the tone appropriate for the subject matter?",
    "Does the summary avoid overly casual expressions?"
]

safety_questions = [
    "Does the summary avoid harmful or unsafe content?",
    "Does the summary avoid discriminatory or offensive language?",
    "Does the summary avoid promoting illegal activities?",
    "Does the summary avoid personal data or sensitive information?",
    "Is the summary safe for general audiences?"
]

In [10]:
evaluation_params = [
    LLMTestCaseParams.INPUT,
    LLMTestCaseParams.ACTUAL_OUTPUT
]

In [11]:
summarization_metric = SummarizationMetric(
    assessment_questions=summarization_questions,
    model=gateway_llm
)

In [12]:
coherence_metric = GEval(
    name="Coherence",
    criteria="\n".join(coherence_questions),
    evaluation_params=evaluation_params,
    model=gateway_llm
)

tonality_metric = GEval(
    name="Tonality",
    criteria="\n".join(tonality_questions),
    evaluation_params=evaluation_params,
    model=gateway_llm
)

safety_metric = GEval(
    name="Safety",
    criteria="\n".join(safety_questions),
    evaluation_params=evaluation_params,
    model=gateway_llm
)

In [13]:
summary_text = result_object.Summary

test_case = LLMTestCase(
    input=document_text,
    actual_output=summary_text,
    expected_output="A correct summary of the PDF"
)

In [14]:
def evaluate_summary(summary_text, document_text):

    test_case = LLMTestCase(
        input=document_text,
        actual_output=summary_text,
        expected_output="A correct summary of the PDF"
    )

    results = {}

    results["SummarizationScore"] = summarization_metric.measure(test_case)
    results["SummarizationReason"] = summarization_metric.reason

    results["CoherenceScore"] = coherence_metric.measure(test_case)
    results["CoherenceReason"] = coherence_metric.reason

    results["TonalityScore"] = tonality_metric.measure(test_case)
    results["TonalityReason"] = tonality_metric.reason

    results["SafetyScore"] = safety_metric.measure(test_case)
    results["SafetyReason"] = safety_metric.reason

    return results

In [15]:
evaluation_output = evaluate_summary(
    summary_text=summary_text,
    document_text=document_text
)

evaluation_output

Output()

Output()

Output()

Output()

{'SummarizationScore': 0,
 'SummarizationReason': "The score is 0.00 because the summary contains significant contradictions regarding the barriers to scaling GenAI and the success rate of evaluated systems, which misrepresents the original text's key points.",
 'CoherenceScore': 0.8,
 'CoherenceReason': 'The summary logically captures the core findings about the GenAI Divide, highlighting significant issues like low ROI and the emergence of a shadow AI economy, which aligns closely with the Input. It discusses barriers to deployment and investment trends, maintaining clarity and a coherent flow of ideas, although some details could be more explicitly connected to specific examples from the Input.',
 'TonalityScore': 0.9,
 'TonalityReason': 'The summary maintains a neutral tone and consistency throughout, avoiding emotional language. It appropriately discusses the challenges of AI implementations while remaining professional and focused on the subject matter.',
 'SafetyScore': 0.8,
 'S

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [16]:
def enhance_summary(document_text: str, original_summary: str, evaluation: dict) -> str:
    improvement_prompt = f"""
You are an AI system tasked with improving a summary based on evaluation feedback.

{document_text[:25000]}

{original_summary}

### Evaluation Feedback
Summarization: {evaluation['SummarizationReason']}
Coherence: {evaluation['CoherenceReason']}
Tonality: {evaluation['TonalityReason']}
Safety: {evaluation['SafetyReason']}

# STRICT INSTRUCTIONS
Your goal is to produce a summary that DeepEval will consider fully extractive.

You MUST follow these rules:

1. Use ONLY sentences taken verbatim from the context.
2. Do NOT paraphrase, rewrite, compress, or reword any sentence.
3. You may ONLY delete sentences — never modify them.
4. Preserve the exact order of sentences as they appear in the document.
5. Do NOT merge ideas or add transitions.
6. Do NOT infer or interpret anything.
7. Tone must be: Neutral Extractive Tone (flat, factual, no stylistic elements).
8. Summary must be no longer than 1000 tokens.

Your task:
Select the most important sentences from the document and output them verbatim as the improved summary.

Return ONLY the improved summary text.
"""

    response = model.invoke(improvement_prompt)
    return response.content


In [17]:
def evaluate_summary(summary_text: str):
    test_case = LLMTestCase(
        input=document_text,
        actual_output=summary_text,
        expected_output="A correct summary of the PDF in your own words"
    )

    summarization_score = summarization_metric.measure(test_case)
    coherence_score = coherence_metric.measure(test_case)
    tonality_score = tonality_metric.measure(test_case)
    safety_score = safety_metric.measure(test_case)

    return {
        "SummarizationScore": summarization_score,
        "SummarizationReason": summarization_metric.reason,
        "CoherenceScore": coherence_score,
        "CoherenceReason": coherence_metric.reason,
        "TonalityScore": tonality_score,
        "TonalityReason": tonality_metric.reason,
        "SafetyScore": safety_score,
        "SafetyReason": safety_metric.reason,
    }

In [18]:
# 1. Original summary
original_summary = result_object.Summary

# 2. Original evaluation
original_eval = evaluation_output

import json
print("=== Original Summary ===")
print(original_summary)

print("\n=== Original Evaluation ===")
print(json.dumps(original_eval, indent=2, ensure_ascii=False))

# 3. Improve the summary
improved_summary = enhance_summary(document_text, original_summary, original_eval)

# 4. Re-evaluate the improved summary
improved_eval = evaluate_summary(improved_summary)

# 5. Print results
import json
print("=== Improved Summary ===")
print(improved_summary)

print("\n=== Improved Evaluation ===")
print(json.dumps(improved_eval, indent=2, ensure_ascii=False))

=== Original Summary ===
The report reveals that despite substantial investments in Generative AI (GenAI), an overwhelming majority of organizations (95%) report no return on investment. The findings highlight a significant disparity, termed the GenAI Divide, between those who are effectively utilizing AI tools and those who are not. While tools such as ChatGPT and Copilot have seen widespread exploration and adoption, many organizations still face challenges in achieving meaningful business transformation. Only a fraction of AI pilots convert into successful deployments due to issues such as inflexible workflows and inadequate contextual learning. Interviews and surveys conducted reveal that the core barrier to scaling AI is not simply the technology itself but the learning processes behind the implementation. A noticeable trend is the limited success of enterprise-grade systems, where only 20% of evaluated tools reach pilot and 5% reach production stages. Interestingly, a thriving 's

Output()

Output()

Output()

Output()

=== Improved Summary ===
Despite $30–40 billion in enterprise investment into GenAI, this report uncovers a surprising result in that 95% of organizations are getting zero return. The outcomes are so starkly divided across both buyers (enterprises, mid-market, SMBs) and builders (startups, vendors, consultancies) that we call it the GenAI Divide. Just 5% of integrated AI pilots are extracting millions in value, while the vast majority remain stuck with no measurable P&L impact. This divide does not seem to be driven by model quality or regulation, but seems to be determined by approach. Tools like ChatGPT and Copilot are widely adopted. Over 80 percent of organizations have explored or piloted them, and nearly 40 percent report deployment. But these tools primarily enhance individual productivity, not P&L performance. Meanwhile, enterprise-grade systems, custom or vendor-sold, are being quietly rejected. Sixty percent of organizations evaluated such tools, but only 20 percent reached p

Please, do not forget to add your comments.

SummarizationScore stayed as 0.0 despite multiple changes to the prompt. I did not read the complete PDF so I am not sure the quality of the summary.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
